# Notebook 03 — Description Enrichment
Scrapes StoryGraph to fill missing descriptions for books collected in Notebook 01.
Also adds genres, moods, pace, and rating from StoryGraph.
Output: `Data/Raw/non_western_fiction/collected.json`

In [7]:
# ── Notebook 03 — Merge GR + OL ────────────────────────────────────────────
import json, re
import pandas as pd
from pathlib import Path

REPO_ROOT  = Path().resolve().parents[1]
RAW_DIR    = REPO_ROOT / "Data" / "Raw" / "non_western_fantasy"
CLEAN_DIR  = REPO_ROOT / "Data" / "Clean"
CLEAN_DIR.mkdir(exist_ok=True)

ol_raw = pd.DataFrame(json.load(open(RAW_DIR / "ol_genre_first.json")))
gr_raw = pd.read_csv(RAW_DIR / "goodreads_with_descriptions.csv")

print(f"✅ Loaded")
print(f"   OL books:         {len(ol_raw):,}")
print(f"   Goodreads books:  {len(gr_raw):,}")
print(f"\n   OL columns:  {list(ol_raw.columns)}")
print(f"   GR columns:  {list(gr_raw.columns)}")

✅ Loaded
   OL books:         4,364
   Goodreads books:  2,129

   OL columns:  ['title', 'author', 'year_published', 'cover_url', 'avg_rating', 'num_ratings', 'subjects', 'source', 'source_url', 'source_tag', 'query', 'description']
   GR columns:  ['title', 'author', 'goodreads_url', 'cover_url', 'avg_rating', 'num_ratings', 'year_published', 'shelf', 'description']


In [8]:
# ── Cell 2 — Filter OL noise ────────────────────────────────────────────────
WESTERN_NOISE_AUTHORS = {
    "edgar rice burroughs", "h. rider haggard", "rudyard kipling",
    "joseph conrad", "wilbur smith", "james michener", "john buchan",
    "arthur conan doyle", "h.g. wells", "jules verne",
}

NOISE_SUBJECTS = {
    "tarzan", "colonial", "colonialism", "imperialism",
    "safari", "big game hunting", "missionaries",
}

CULTURAL_SIGNALS = {
    "yoruba", "igbo", "akan", "zulu", "xhosa", "orisha", "anansi",
    "afrofuturism", "africanfuturism", "african fantasy",
    "yokai", "kami", "oni", "kitsune", "tengu", "wuxia", "xianxia",
    "cultivation", "jade emperor", "chinese fantasy", "japanese fantasy",
    "korean fantasy", "manhwa", "manhua",
    "djinn", "jinn", "ifrit", "persian", "ottoman", "mughal", "arabian",
    "hindu", "vedic", "mahabharata", "ramayana", "rakshasa", "asura",
    "aztec", "mayan", "inca", "nahua", "mesoamerican", "latin american fantasy",
    "curandera", "brujeria",
    "native american", "indigenous", "navajo", "lakota", "ojibwe",
    "anishinaabe", "first nations",
    "maori", "aboriginal", "polynesian", "pacific islander",
    "filipino", "aswang", "diwata", "babaylan",
    "vietnamese", "thai fantasy",
    "magical realism", "speculative fiction",
}

TRUSTED_QUERIES = {
    "afrofuturism", "africanfuturism", "wuxia", "xianxia",
    "fantasy yoruba", "fantasy igbo", "fantasy akan", "fantasy orisha",
    "fantasy anansi", "fantasy kitsune", "fantasy yokai", "fantasy aswang",
    "fantasy djinn", "fantasy mahabharata", "fantasy ramayana", "fantasy aztec",
    "fantasy curandera",
}

def ol_passes_filter(row):
    if str(row.get("author", "")).lower().strip() in WESTERN_NOISE_AUTHORS:
        return False
    subjects_text = " ".join(str(s) for s in (row.get("subjects") or [])).lower()
    if any(n in subjects_text for n in NOISE_SUBJECTS):
        return False
    if row.get("query") in TRUSTED_QUERIES:
        return True
    text = subjects_text + " " + str(row.get("title", "")).lower()
    return any(signal in text for signal in CULTURAL_SIGNALS)

mask        = ol_raw.apply(ol_passes_filter, axis=1)
ol_filtered = ol_raw[mask].copy().reset_index(drop=True)
ol_removed  = ol_raw[~mask].copy().reset_index(drop=True)

print(f"── OL filter results ──")
print(f"  Before:  {len(ol_raw):,}")
print(f"  After:   {len(ol_filtered):,}")
print(f"  Removed: {len(ol_removed):,}")
print(f"\n  Sample removed (first 10):")
for _, r in ol_removed.head(10).iterrows():
    print(f"    {r['title']} — {r['author']}")

── OL filter results ──
  Before:  4,364
  After:   1,079
  Removed: 3,285

  Sample removed (first 10):
    Tarzan of the Apes — Edgar Rice Burroughs
    She — H. Rider Haggard
    King Solomon's Mines — H. Rider Haggard
    The Secret Agent — Joseph Conrad
    Cinq semaines en ballon — Jules Verne
    The Return of Tarzan — Edgar Rice Burroughs
    The People of the Mist — H. Rider Haggard
    The Grey Fairy Book — Andrew Lang
    Tarzan the Untamed (Book #7) — Edgar Rice Burroughs
    The Beasts of Tarzan (Classic Ace SF, F-203) — Edgar Rice Burroughs


In [9]:
# ── Cell 3 — Audit removed books ───────────────────────────────────────────
print("Sample removed books by query tag:\n")

for tag in ol_removed["source_tag"].unique():
    sample = ol_removed[ol_removed["source_tag"] == tag].head(5)
    print(f"── {tag} ──")
    for _, r in sample.iterrows():
        subj_preview = str(r["subjects"])[:80] if r["subjects"] else "no subjects"
        print(f"  {r['title']} — {r['author']}")
        print(f"    subjects: {subj_preview}")
    print()

Sample removed books by query tag:

── africa ──
  Tarzan of the Apes — Edgar Rice Burroughs
    subjects: ['British in fiction', 'Historical Fiction', 'Wild men in fiction', 'American Ad
  She — H. Rider Haggard
    subjects: ['Fiction in English', 'Adventure and adventurers', 'Reincarnation', 'Fiction in
  King Solomon's Mines — H. Rider Haggard
    subjects: ["Children's fiction", 'Mines and mineral resources, fiction', 'Adventure and ad
  The Secret Agent — Joseph Conrad
    subjects: ['Anarchists, fiction', 'Conspiracies, fiction', 'Bombings, fiction', 'Terrorism
  Cinq semaines en ballon — Jules Verne
    subjects: ['Travel', 'Hot air balloons', 'Description and travel', 'Fiction', 'Classic Lit

── zulu ──
  Black Heart and White Heart — H. Rider Haggard
    subjects: ['Classic Literature', 'Fiction', 'Fiction, romance, general', 'Fiction, fantasy

── anansi ──
  LiTTscapes - Landscapes of Fiction from Trinidad and Tobago — Kris Rampersad
    subjects: ['Literature', 'Fiction', '

In [10]:
# ── Cell 4 — Smarter filter ─────────────────────────────────────────────────

# These broad queries return too much western noise — don't trust them alone
NOISY_QUERIES = {
    "fantasy korean", "fantasy indian", "fantasy india",
    "fantasy chinese", "fantasy japan", "fantasy japanese",
    "fantasy african", "fantasy africa", "science fiction africa",
    "fantasy latin american", "science fiction latin american",
    "fantasy native american", "fantasy indigenous",
    "fantasy arabian", "fantasy persian", "fantasy islamic",
    "fantasy vietnamese", "fantasy thai", "fantasy philippine",
    "fantasy aboriginal", "fantasy pacific islander",
    "fantasy maori", "fantasy polynesian",
    "fantasy maya", "fantasy mayan", "fantasy inca", "fantasy aztec",
    "fantasy zulu", "fantasy navajo", "fantasy lakota",
}

# These specific queries are reliable — trust them without needing subject signals
TRUSTED_QUERIES = {
    "afrofuturism", "africanfuturism", "wuxia", "xianxia",
    "fantasy yoruba", "fantasy igbo", "fantasy akan", "fantasy orisha",
    "fantasy anansi", "fantasy kitsune", "fantasy yokai", "fantasy aswang",
    "fantasy djinn", "fantasy mahabharata", "fantasy ramayana",
    "fantasy aztec", "fantasy curandera", "fantasy cultivation",
    "fantasy filipino", "fantasy vietnamese", "fantasy thai",
    "fantasy philippine", "fantasy ottoman", "fantasy mughal",
    "fantasy aboriginal", "fantasy maori", "fantasy pacific islander",
    "fantasy polynesian", "fantasy navajo", "fantasy lakota",
    "fantasy persian", "fantasy arabian", "fantasy islamic",
}

WESTERN_NOISE_AUTHORS = {
    "edgar rice burroughs", "h. rider haggard", "rudyard kipling",
    "joseph conrad", "wilbur smith", "james michener", "john buchan",
    "arthur conan doyle", "h.g. wells", "jules verne",
    "charlotte perkins gilman", "george macdonald", "d. h. lawrence",
    "lucy maud montgomery", "carlo collodi", "neil gaiman",
    "george r. r. martin", "rick riordan", "j. k. rowling",
    "bram stoker", "edith nesbit", "l. frank baum",
    "richard adams", "roald dahl", "c. s. lewis",
    "stephen king", "astrid lindgren", "bill willingham",
}

NOISE_SUBJECTS = {
    "tarzan", "colonialism", "imperialism",
    "safari", "big game hunting", "missionaries",
}

CULTURAL_SIGNALS = {
    "yoruba", "igbo", "akan", "zulu", "xhosa", "orisha", "anansi",
    "afrofuturism", "africanfuturism", "african fantasy", "african fiction",
    "yokai", "kami", "oni", "kitsune", "tengu", "wuxia", "xianxia",
    "cultivation", "jade emperor", "chinese fantasy", "japanese fantasy",
    "korean fantasy", "manhwa", "manhua", "manga",
    "djinn", "jinn", "ifrit", "persian", "ottoman", "mughal", "arabian",
    "hindu", "vedic", "mahabharata", "ramayana", "rakshasa", "asura",
    "aztec", "mayan", "inca", "nahua", "mesoamerican",
    "curandera", "brujeria", "latin american",
    "native american", "indigenous", "navajo", "lakota", "ojibwe",
    "anishinaabe", "first nations",
    "maori", "aboriginal", "polynesian", "pacific islander",
    "filipino", "aswang", "diwata", "babaylan",
    "vietnamese", "thai", "korean", "japanese", "chinese",
    "magical realism", "speculative fiction", "afrofuturism",
    "african", "south asian", "southeast asian",
}

def ol_passes_filter(row):
    author = str(row.get("author", "")).lower().strip()
    query  = row.get("query", "")

    # Always remove known western noise authors
    if author in WESTERN_NOISE_AUTHORS:
        return False

    subjects_text = " ".join(str(s) for s in (row.get("subjects") or [])).lower()

    # Remove noise subjects regardless of query
    if any(n in subjects_text for n in NOISE_SUBJECTS):
        return False

    # Trusted specific queries — keep unless western author
    if query in TRUSTED_QUERIES:
        return True

    # For noisy broad queries — require cultural signal in subjects or title
    if query in NOISY_QUERIES:
        text = subjects_text + " " + str(row.get("title", "")).lower()
        return any(signal in text for signal in CULTURAL_SIGNALS)

    # Default: keep
    return True

mask        = ol_raw.apply(ol_passes_filter, axis=1)
ol_filtered = ol_raw[mask].copy().reset_index(drop=True)
ol_removed  = ol_raw[~mask].copy().reset_index(drop=True)

print(f"── OL filter results ──")
print(f"  Before:  {len(ol_raw):,}")
print(f"  After:   {len(ol_filtered):,}")
print(f"  Removed: {len(ol_removed):,}")
print(f"\n  Removed by source_tag:")
for tag, count in ol_removed["source_tag"].value_counts().items():
    print(f"    {tag:25s}: {count:,}")
print(f"\n  Kept by source_tag:")
for tag, count in ol_filtered["source_tag"].value_counts().items():
    print(f"    {tag:25s}: {count:,}")

── OL filter results ──
  Before:  4,364
  After:   2,734
  Removed: 1,630

  Removed by source_tag:
    japanese                 : 553
    south_asian              : 396
    africa                   : 372
    latin_american           : 143
    chinese                  : 123
    korean                   : 16
    indigenous_americas      : 10
    southeast_asian          : 4
    oceania                  : 4
    anansi                   : 3
    filipino                 : 2
    middle_eastern           : 2
    zulu                     : 1
    afrofuturism             : 1

  Kept by source_tag:
    japanese                 : 1,230
    chinese                  : 510
    africa                   : 263
    middle_eastern           : 202
    latin_american           : 128
    south_asian              : 79
    korean                   : 74
    afrofuturism             : 58
    wuxia                    : 43
    southeast_asian          : 35
    filipino                 : 28
    oceania          

In [11]:
# ── Quick sanity check ──────────────────────────────────────────────────────
check_titles = ["Spice Road", "Serpent Sea", "The Sword of Kaigen", "Bleach", "Naruto"]
for t in check_titles:
    found = ol_filtered[ol_filtered["title"].str.contains(t, case=False, na=False)]
    status = "✅ KEPT" if len(found) > 0 else "❌ REMOVED"
    print(f"  {status}: {t}")

  ✅ KEPT: Spice Road
  ✅ KEPT: Serpent Sea
  ❌ REMOVED: The Sword of Kaigen
  ✅ KEPT: Bleach
  ✅ KEPT: Naruto


In [12]:
# ── Cell 5 — Remove comics/manga/graphic novels + fix edge cases ────────────

COMIC_SUBJECTS = {
    "graphic novel", "graphic novels", "comics", "comic book",
    "manga", "manhwa", "manhua", "comic strip",
}

COMIC_TITLE_SIGNALS = [
    "vol.", "volume", "#1", "#2", "#3", "issue", "omnibus",
]

def is_comic(row):
    subjects_text = " ".join(str(s) for s in (row.get("subjects") or [])).lower()
    if any(c in subjects_text for c in COMIC_SUBJECTS):
        return True
    # Catch manga/comic series by title patterns
    title_lower = str(row.get("title", "")).lower()
    if any(sig in title_lower for sig in COMIC_TITLE_SIGNALS):
        return True
    return False

# Also add M.L. Wang to a manual keeplist — good book, sparse OL subjects
MANUAL_KEEP_TITLES = {
    "the sword of kaigen",
}

def final_filter(row):
    if str(row.get("title", "")).lower().strip() in MANUAL_KEEP_TITLES:
        return True
    if is_comic(row):
        return False
    return True

mask2       = ol_filtered.apply(final_filter, axis=1)
ol_clean    = ol_filtered[mask2].copy().reset_index(drop=True)
ol_removed2 = ol_filtered[~mask2].copy().reset_index(drop=True)

print(f"── Comic/manga filter ──")
print(f"  Before:  {len(ol_filtered):,}")
print(f"  After:   {len(ol_clean):,}")
print(f"  Removed: {len(ol_removed2):,}")
print(f"\n  Sample removed:")
for _, r in ol_removed2.head(10).iterrows():
    print(f"    {r['title']} — {r['author']}")

# Verify
for t in ["The Sword of Kaigen", "Bleach", "Naruto", "Spice Road"]:
    found = ol_clean[ol_clean["title"].str.contains(t, case=False, na=False)]
    print(f"  {'✅' if len(found) else '❌'} {t}")

── Comic/manga filter ──
  Before:  2,734
  After:   1,647
  Removed: 1,087

  Sample removed:
    Hotel Africa Volume 2 (Hotel Africa) — Hee Jung Park
    Hotel Africa Volume 1 — Hee Jung Park
    Mobile suit Gundam seed — Iwase, Masatsugu.
    Rapport final — Séminaire Africana 90 "L'imaginaire dans le théâtre et la b.d. africaine" (1990 La Chaux-de-Fonds, Switzerland)
    Frankenstein or The Modern Prometheus — Mary Shelley
    Keeping It Unreal — Darieck Scott
    Marwe: Into the Land of the Dead [An East African Legend] (Graphic Myths and Legends) — Marie P. Croall
    Picking Up The Ghost — Tone Milazzo
    Bayou Volume Two — Jeremy Love
    Bayou — Jeremy Love
  ❌ The Sword of Kaigen
  ❌ Bleach
  ❌ Naruto
  ✅ Spice Road


In [14]:
# ── Cell 5 — Fixed version ──────────────────────────────────────────────────

COMIC_SUBJECTS = {
    "graphic novel", "graphic novels", "comics", "comic book",
    "comic strip", "sequential art",
}

# Only remove by title pattern if it also has comic subjects
# (vol.1 alone is not enough — "The Sword of Kaigen Vol.1" is still a novel)
MANUAL_KEEP_TITLES = {
    "the sword of kaigen",
}

MANUAL_REMOVE_AUTHORS = {
    "mary shelley",  # Frankenstein — not relevant
}

def is_comic(row):
    title_lower   = str(row.get("title", "")).lower().strip()
    subjects_text = " ".join(str(s) for s in (row.get("subjects") or [])).lower()

    # Manual keep always wins
    if title_lower in MANUAL_KEEP_TITLES:
        return False

    # Manual remove authors
    if str(row.get("author", "")).lower().strip() in MANUAL_REMOVE_AUTHORS:
        return True

    # Only flag as comic if subjects explicitly say so
    if any(c in subjects_text for c in COMIC_SUBJECTS):
        return True

    return False

ol_clean    = ol_filtered[~ol_filtered.apply(is_comic, axis=1)].copy().reset_index(drop=True)
ol_removed2 = ol_filtered[ol_filtered.apply(is_comic, axis=1)].copy().reset_index(drop=True)

print(f"── Comic filter ──")
print(f"  Before:  {len(ol_filtered):,}")
print(f"  After:   {len(ol_clean):,}")
print(f"  Removed: {len(ol_removed2):,}")
print(f"\n  Sample removed:")
for _, r in ol_removed2.head(10).iterrows():
    print(f"    {r['title']} — {r['author']}")

print()
for t in ["The Sword of Kaigen", "Bleach", "Naruto", "Spice Road", "Frankenstein"]:
    found = ol_clean[ol_clean["title"].str.contains(t, case=False, na=False)]
    print(f"  {'✅ KEPT' if len(found) else '❌ REMOVED'}: {t}")

── Comic filter ──
  Before:  2,734
  After:   1,679
  Removed: 1,055

  Sample removed:
    Hotel Africa Volume 2 (Hotel Africa) — Hee Jung Park
    Hotel Africa Volume 1 — Hee Jung Park
    Mobile suit Gundam seed — Iwase, Masatsugu.
    Rapport final — Séminaire Africana 90 "L'imaginaire dans le théâtre et la b.d. africaine" (1990 La Chaux-de-Fonds, Switzerland)
    Frankenstein or The Modern Prometheus — Mary Shelley
    Keeping It Unreal — Darieck Scott
    Marwe: Into the Land of the Dead [An East African Legend] (Graphic Myths and Legends) — Marie P. Croall
    Picking Up The Ghost — Tone Milazzo
    Bayou Volume Two — Jeremy Love
    Bayou — Jeremy Love

  ❌ REMOVED: The Sword of Kaigen
  ❌ REMOVED: Bleach
  ❌ REMOVED: Naruto
  ✅ KEPT: Spice Road
  ❌ REMOVED: Frankenstein


In [15]:
# ── Debug Sword of Kaigen ───────────────────────────────────────────────────
sword = ol_raw[ol_raw["title"].str.contains("Sword of Kaigen", case=False, na=False)]
for _, r in sword.iterrows():
    print(f"Title:    {r['title']}")
    print(f"Author:   {r['author']}")
    print(f"Subjects: {r['subjects']}")
    print(f"Query:    {r['query']}")

Title:    The Sword of Kaigen
Author:   M. L. Wang
Subjects: ['Fiction', 'Fiction, fantasy, general', 'Fantasy', 'High Fantasy', 'Fiction, fantasy, epic', 'FICTION / Fantasy / Epic', 'Epic Fantasy', 'Magic', 'Science Fiction & Fantasy']
Query:    fantasy maya


In [16]:
# ── Cell 5 — Final filter logic ─────────────────────────────────────────────

WESTERN_NOISE_AUTHORS = {
    "edgar rice burroughs", "h. rider haggard", "rudyard kipling",
    "joseph conrad", "wilbur smith", "james michener", "john buchan",
    "arthur conan doyle", "h.g. wells", "jules verne",
    "charlotte perkins gilman", "george macdonald", "d. h. lawrence",
    "lucy maud montgomery", "carlo collodi", "neil gaiman",
    "george r. r. martin", "rick riordan", "j. k. rowling",
    "bram stoker", "edith nesbit", "l. frank baum",
    "richard adams", "roald dahl", "c. s. lewis",
    "stephen king", "astrid lindgren", "bill willingham",
    "mary shelley", "niccolò machiavelli", "michael ende",
    "charles kingsley", "andrew lang",
}

NOISE_SUBJECTS = {
    "tarzan", "colonialism", "imperialism",
    "safari", "big game hunting", "missionaries",
}

COMIC_SUBJECTS = {
    "graphic novel", "graphic novels", "comics", "comic book",
    "comic strip", "sequential art",
}

def ol_final_filter(row):
    author        = str(row.get("author", "")).lower().strip()
    subjects_text = " ".join(str(s) for s in (row.get("subjects") or [])).lower()

    # Remove known western authors
    if author in WESTERN_NOISE_AUTHORS:
        return False

    # Remove noise subjects
    if any(n in subjects_text for n in NOISE_SUBJECTS):
        return False

    # Remove comics/graphic novels
    if any(c in subjects_text for c in COMIC_SUBJECTS):
        return False

    return True

ol_clean    = ol_raw[ol_raw.apply(ol_final_filter, axis=1)].copy().reset_index(drop=True)
ol_removed  = ol_raw[~ol_raw.apply(ol_final_filter, axis=1)].copy().reset_index(drop=True)

print(f"── Final OL filter ──")
print(f"  Before:  {len(ol_raw):,}")
print(f"  After:   {len(ol_clean):,}")
print(f"  Removed: {len(ol_removed):,}")
print(f"\n  Kept by source_tag:")
for tag, count in ol_clean["source_tag"].value_counts().items():
    print(f"    {tag:25s}: {count:,}")

print()
for t in ["The Sword of Kaigen", "Spice Road", "Tarzan", "Harry Potter", "Naruto"]:
    found = ol_clean[ol_clean["title"].str.contains(t, case=False, na=False)]
    print(f"  {'✅ KEPT' if len(found) else '❌ REMOVED'}: {t}")

── Final OL filter ──
  Before:  4,364
  After:   2,884
  Removed: 1,480

  Kept by source_tag:
    japanese                 : 607
    africa                   : 546
    chinese                  : 525
    south_asian              : 440
    latin_american           : 256
    middle_eastern           : 189
    korean                   : 73
    afrofuturism             : 58
    wuxia                    : 43
    southeast_asian          : 32
    filipino                 : 26
    indigenous_americas      : 23
    oceania                  : 22
    xianxia                  : 21
    anansi                   : 13
    igbo                     : 3
    orisha                   : 3
    zulu                     : 2
    yoruba                   : 1
    akan                     : 1

  ✅ KEPT: The Sword of Kaigen
  ✅ KEPT: Spice Road
  ❌ REMOVED: Tarzan
  ✅ KEPT: Harry Potter
  ❌ REMOVED: Naruto


In [17]:
# ── Who are the most common authors in ol_clean? ───────────────────────────
author_counts = ol_clean["author"].value_counts()

print("Top 100 authors in filtered OL data:")
print(author_counts.head(100).to_string())

Top 100 authors in filtered OL data:
author
                              107
Joseph Li                      13
Maya Daniels                   13
L. A. Banks                    10
Dr. Seuss                      10
村上春樹                           10
Daisy Meadows                  10
Tao Wong                       10
Lynne Reid Banks               10
Vicki Corona                   10
Gillian Rubinstein              9
Octavia E. Butler               8
Reggie ROBINSON                 8
Edgar Allan Poe                 7
Lian Hearn                      7
Erica BAKER                     7
Hasaway Anime Corner            6
Scott Walker                    6
Philip Kerr                     6
Tara Maya                       6
Erin Hunter                     5
Lauren Beukes                   5
Samuel R. Delany                5
Piers Anthony                   5
Nalo Hopkinson                  5
A. J. Hartley                   5
Rochelle Alers                  5
Mary Pope Osborne               5
Lewi

In [24]:
# ── Cell 6 — Updated western author blocklist ───────────────────────────────
WESTERN_NOISE_AUTHORS = {
    # Original list
    "edgar rice burroughs", "h. rider haggard", "rudyard kipling",
    "joseph conrad", "wilbur smith", "james michener", "john buchan",
    "arthur conan doyle", "h.g. wells", "jules verne",
    "charlotte perkins gilman", "george macdonald", "d. h. lawrence",
    "lucy maud montgomery", "carlo collodi", "neil gaiman",
    "george r. r. martin", "rick riordan", "j. k. rowling",
    "bram stoker", "edith nesbit", "l. frank baum",
    "richard adams", "roald dahl", "c. s. lewis",
    "stephen king", "astrid lindgren", "bill willingham",
    "mary shelley", "niccolò machiavelli", "michael ende",
    "charles kingsley", "andrew lang",
    # New additions from top 100 audit
    "dr. seuss", "lewis carroll", "edgar allan poe",
    "nathaniel hawthorne", "mark twain", "antoine de saint-exupéry",
    "christopher paolini", "j.r.r. tolkien", "mercedes lackey",
    "piers anthony", "mary pope osborne", "daisy meadows",
    "erin hunter", "paul theroux", "harold macgrath",
    "charles de lint", "james rollins",
}

NOISE_SUBJECTS = {
    "tarzan", "colonialism", "imperialism",
    "safari", "big game hunting", "missionaries",
}

COMIC_SUBJECTS = {
    "graphic novel", "graphic novels", "comics", "comic book",
    "comic strip", "sequential art",
}

# ── Add to filter: remove non-fiction / academic books ─────────────────────
NONFICTION_SUBJECTS = {
    "criticism", "critical essays", "literary criticism",
    "history and criticism", "congresses", "study and teaching",
    "nonfiction", "biography", "autobiography",
    "education", "philosophy", "sociology", "political science",
    "essays", "handbooks", "reference",
    "cross-cultural studies", "mass media", "self-perception",
    "creative ability in children", "imagination in children",
}

def ol_final_filter(row):
    author        = str(row.get("author", "")).lower().strip()
    subjects_text = " ".join(str(s) for s in (row.get("subjects") or [])).lower()

    if author in WESTERN_NOISE_AUTHORS:
        return False
    if any(n in subjects_text for n in NOISE_SUBJECTS):
        return False
    if any(c in subjects_text for c in COMIC_SUBJECTS):
        return False
    if any(n in subjects_text for n in NONFICTION_SUBJECTS):
        return False
    return True

ol_clean   = ol_raw[ol_raw.apply(ol_final_filter, axis=1)].copy().reset_index(drop=True)
ol_removed = ol_raw[~ol_raw.apply(ol_final_filter, axis=1)].copy().reset_index(drop=True)

print(f"── Final OL filter ──")
print(f"  Before:  {len(ol_raw):,}")
print(f"  After:   {len(ol_clean):,}")
print(f"  Removed: {len(ol_removed):,}")

print()
for t in ["Harry Potter", "Tarzan", "The Sword of Kaigen",
          "Spice Road", "Children of Blood and Bone"]:
    found = ol_clean[ol_clean["title"].str.contains(t, case=False, na=False)]
    status = f"✅ KEPT ({len(found)})" if len(found) else "❌ REMOVED"
    print(f"  {status}: {t}")

print(f"\nTop 30 authors:")
print(ol_clean["author"].value_counts().head(30).to_string())

── Final OL filter ──
  Before:  4,364
  After:   2,355
  Removed: 2,009

  ❌ REMOVED: Harry Potter
  ❌ REMOVED: Tarzan
  ✅ KEPT (1): The Sword of Kaigen
  ✅ KEPT (1): Spice Road
  ✅ KEPT (1): Children of Blood and Bone

Top 30 authors:
author
                        99
Joseph Li               13
Maya Daniels            13
L. A. Banks             10
村上春樹                    10
Tao Wong                10
Lynne Reid Banks        10
Vicki Corona            10
Gillian Rubinstein       9
Octavia E. Butler        8
Reggie ROBINSON          8
Lian Hearn               7
Erica BAKER              7
Hasaway Anime Corner     6
Scott Walker             6
Philip Kerr              6
Tara Maya                6
Lauren Beukes            5
Samuel R. Delany         5
Nalo Hopkinson           5
A. J. Hartley            5
Rochelle Alers           5
Yuan, Mei                5
Henriette Barkow         5
Yoshitaka Amano          5
Steve Bein               5
Mo Xiang Tong Xiu        5
Jami Fairleigh           5


In [20]:
# ── Quick debug ─────────────────────────────────────────────────────────────
hp = ol_clean[ol_clean["title"].str.contains("Harry Potter", case=False, na=False)]
print("Harry Potter entries:")
for _, r in hp.iterrows():
    print(f"  author: '{r['author']}' | query: {r['query']}")

Harry Potter entries:
  author: 'Maya Götz' | query: fantasy maya
  author: 'Maya Götz' | query: fantasy maya
  author: 'Elizabeth E. Heilman' | query: fantasy navajo


In [23]:
hp = ol_clean[ol_clean["title"].str.contains("Harry Potter", case=False, na=False)]
for _, r in hp.iterrows():
    print(f"Title:    {r['title']}")
    print(f"Author:   {r['author']}")
    print(f"Subjects: {r['subjects']}")
    print(f"Query:    {r['query']}")
    print()

Title:    Mit Pokémon in Harry Potters Welt
Author:   Maya Götz
Subjects: ['Cross-cultural studies', 'Imagination in children', 'Role models', 'Mass media and children', 'Fantasy in children', 'Self-perception in children', 'Creative ability in children']
Query:    fantasy maya

Title:    Mit Pokémon in Harry Potters Welt
Author:   Maya Götz
Subjects: ['Cross-cultural studies', 'Imagination in children', 'Role models', 'Mass media and children', 'Fantasy in children', 'Self-perception in children', 'Creative ability in children']
Query:    fantasy maya



In [25]:
# ── Check non-English titles ────────────────────────────────────────────────
from langdetect import detect

def is_english_title(title):
    try:
        return detect(str(title)) == "en"
    except:
        return True

non_english = ol_clean[~ol_clean["title"].apply(is_english_title)]
print(f"Non-English titles: {len(non_english):,}")
print(f"\nSample:")
for _, r in non_english.head(20).iterrows():
    print(f"  {r['title']} — {r['author']}")

Non-English titles: 947

Sample:
  Akata witch — Nnedi Okorafor
  Coloring Book — animailloverit cor
  Aesheba — 
  Beautiful Skin — Vlad Dima
  Beyond Fantasy — Marialuisa Marino
  Egyptomania — Ronald H. Fritze
  Crota — Rohan M. Vider
  Ape no Evil — Milika M
  Bubbles — Miriam Jordan
  Bugie coloniali — Alberto Alpozzi
  Keystone Passage — Catherine L Quinlan
  Neveryóna — Samuel R. Delany
  Dominion — Zelda Knight
  Mara and Dann — Doris Lessing
  Keystone Passage — Catherine Quinlan
  On The Steel Breeze — Alastair Reynolds
  Kubu Tales - Book 1 — J. Ratcliffe
  Célanire cou-coupé — Maryse Condé
  Devil's valley — André Philippus Brink
  Mojo — Nalo Hopkinson


In [26]:
# ── Check blank author books ────────────────────────────────────────────────
blank = ol_clean[ol_clean["author"].str.strip() == ""]
print(f"Books with no author: {len(blank)}")
print(f"\nSample:")
for _, r in blank.head(15).iterrows():
    print(f"  {r['title']} — query: {r['query']}")

Books with no author: 99

Sample:
  Aesheba — query: fantasy africa
  Collection of 7 fantasy cards on sub-Sahara Africa missionaries — query: fantasy africa
  Animals in Africa — query: fantasy africa
  In and out of Africa — query: fantasy africa
  She — query: fantasy africa
  She — query: fantasy africa
  She — query: fantasy africa
  She — query: fantasy africa
  She — query: fantasy africa
  Zaiire The Prince of Kongo and The Necklace of Destiny — query: fantasy africa
  African Myths and Legends - Folklore for all — query: fantasy african
  THE FUTURE IS BLACK AFROFUTURISM — query: afrofuturism
  Afro-Future Females: Black Writers Chart Science Fiction's Newest New-Wave Trajectory — query: afrofuturism
  The Fantasy Country (3 Volumes) (Chinese Edition) — query: fantasy chinese
  Fantasy in the New Millennium (Chinese Edition) — query: fantasy chinese


In [27]:
# ── Drop blank authors ──────────────────────────────────────────────────────
ol_clean = ol_clean[ol_clean["author"].str.strip() != ""].copy().reset_index(drop=True)
print(f"After dropping blank authors: {len(ol_clean):,}")

After dropping blank authors: 2,256


In [28]:
# ── Cell 7 — Normalize and merge ───────────────────────────────────────────
import re

def norm_title(t):
    t = str(t).lower().strip()
    t = re.sub(r'\s*[\(\[].*?[\)\]]\s*', ' ', t)  # remove (series #1) etc
    t = re.sub(r'\s+', ' ', t).strip()
    return t

def norm_author(a):
    return str(a).lower().strip()

# ── Unified schema for OL ──────────────────────────────────────────────────
ol_unified = pd.DataFrame({
    "title":          ol_clean["title"].str.strip(),
    "author":         ol_clean["author"].fillna(""),
    "description":    ol_clean["description"].fillna(""),
    "year_published": ol_clean["year_published"],
    "cover_url":      ol_clean["cover_url"].fillna(""),
    "avg_rating":     ol_clean["avg_rating"],
    "num_ratings":    ol_clean["num_ratings"],
    "subjects":       ol_clean["subjects"].apply(lambda x: x if isinstance(x, list) else []),
    "source":         "open_library",
    "source_url":     ol_clean["source_url"],
    "source_tag":     ol_clean["source_tag"],
    "title_norm":     ol_clean["title"].apply(norm_title),
    "author_norm":    ol_clean["author"].fillna("").apply(norm_author),
})

# ── Unified schema for Goodreads ───────────────────────────────────────────
gr_unified = pd.DataFrame({
    "title":          gr_raw["title"].str.strip(),
    "author":         gr_raw["author"].fillna(""),
    "description":    gr_raw["description"].fillna(""),
    "year_published": gr_raw["year_published"],
    "cover_url":      gr_raw["cover_url"].fillna(""),
    "avg_rating":     gr_raw["avg_rating"],
    "num_ratings":    gr_raw["num_ratings"],
    "subjects":       [[] for _ in range(len(gr_raw))],
    "source":         "goodreads",
    "source_url":     gr_raw["goodreads_url"],
    "source_tag":     gr_raw["shelf"],
    "title_norm":     gr_raw["title"].apply(norm_title),
    "author_norm":    gr_raw["author"].fillna("").apply(norm_author),
})

# ── Merge — Goodreads wins on overlap ─────────────────────────────────────
gr_lookup = {}
for _, row in gr_unified.iterrows():
    key = (row["title_norm"], row["author_norm"])
    gr_lookup[key] = row

enriched_ol = []
ol_matched  = 0

for _, row in ol_unified.iterrows():
    key      = (row["title_norm"], row["author_norm"])
    gr_match = gr_lookup.get(key)

    if gr_match is not None:
        ol_matched += 1
        row = row.copy()
        if gr_match["description"]:
            row["description"] = gr_match["description"]
        if gr_match["cover_url"]:
            row["cover_url"] = gr_match["cover_url"]
        if pd.notna(gr_match["avg_rating"]):
            row["avg_rating"] = gr_match["avg_rating"]
        if pd.notna(gr_match["num_ratings"]):
            row["num_ratings"] = gr_match["num_ratings"]
        if pd.notna(gr_match["year_published"]):
            row["year_published"] = gr_match["year_published"]
    enriched_ol.append(row)

ol_enriched = pd.DataFrame(enriched_ol)

# Add OL-only books (not already in Goodreads)
gr_keys = set(gr_lookup.keys())
ol_only = ol_enriched[
    ~ol_enriched.apply(lambda r: (r["title_norm"], r["author_norm"]) in gr_keys, axis=1)
]

merged = pd.concat([gr_unified, ol_only], ignore_index=True)
merged = merged.drop(columns=["title_norm", "author_norm"])

print(f"── Merge results ──")
print(f"  Goodreads books:        {len(gr_unified):,}")
print(f"  OL books matched to GR: {ol_matched:,}")
print(f"  OL-only books added:    {len(ol_only):,}")
print(f"  Total merged:           {len(merged):,}")
print(f"\n── Field coverage ──")
for col in ["title","author","description","year_published","cover_url","avg_rating","num_ratings","subjects"]:
    if col == "subjects":
        filled = merged[col].apply(lambda x: isinstance(x, list) and len(x) > 0).sum()
    elif col == "description":
        filled = merged[col].apply(lambda x: bool(str(x).strip()) and str(x).strip() != "nan").sum()
    else:
        filled = merged[col].notna().sum()
    print(f"  {col:20s}: {filled:,}/{len(merged):,} ({filled/len(merged)*100:.0f}%)")

── Merge results ──
  Goodreads books:        2,129
  OL books matched to GR: 103
  OL-only books added:    2,153
  Total merged:           4,282

── Field coverage ──
  title               : 4,282/4,282 (100%)
  author              : 4,282/4,282 (100%)
  description         : 2,108/4,282 (49%)
  year_published      : 3,849/4,282 (90%)
  cover_url           : 4,282/4,282 (100%)
  avg_rating          : 2,249/4,282 (53%)
  num_ratings         : 2,236/4,282 (52%)
  subjects            : 1,739/4,282 (41%)


In [29]:
# ── Cell 8 — Save pre-enrichment file ──────────────────────────────────────
PRE_ENRICH_PATH = CLEAN_DIR / "merged_pre_enrichment.json"

# Convert subjects lists to JSON-safe format
merged_save = merged.copy()
merged_save["subjects"] = merged_save["subjects"].apply(
    lambda x: x if isinstance(x, list) else []
)

merged_save.to_json(PRE_ENRICH_PATH, orient="records", indent=2, force_ascii=False)
print(f"✅ Saved {len(merged_save):,} books to {PRE_ENRICH_PATH}")

# How many need descriptions?
needs_desc = merged[merged["description"].apply(
    lambda x: not bool(str(x).strip()) or str(x).strip() == "nan"
)]
print(f"\nBooks needing descriptions: {len(needs_desc):,}")
print(f"  From OL source: {(needs_desc['source'] == 'open_library').sum():,}")
print(f"  From GR source: {(needs_desc['source'] == 'goodreads').sum():,}")

✅ Saved 4,282 books to C:\Users\Ready2Use\Documents\my-folder\Ironhack-week10\Book-recommendations\Data\Clean\merged_pre_enrichment.json

Books needing descriptions: 2,174
  From OL source: 2,153
  From GR source: 21


In [ ]:
# ── Cell 9 — Google Books enrichment ───────────────────────────────────────
import os, time, json, requests
from dotenv import load_dotenv
from langdetect import detect
from tqdm.notebook import tqdm

load_dotenv()
GOOGLE_BOOKS_API_KEY = os.getenv("GOOGLE_BOOKS_API_KEY", "")
print(f"API key loaded: {'✅' if GOOGLE_BOOKS_API_KEY else '❌ missing'}")

GB_CKPT_PATH = CLEAN_DIR / "google_books_checkpoint.json"

def fetch_google_books_description(title, author=""):
    query  = f"{title} {author}".strip()
    url    = "https://www.googleapis.com/books/v1/volumes"
    params = {
        "q":          query,
        "maxResults": 5,
        "key":        GOOGLE_BOOKS_API_KEY,
        "fields":     "items(volumeInfo(title,authors,description,publishedDate,imageLinks))"
    }
    try:
        resp  = requests.get(url, params=params, timeout=10)
        resp.raise_for_status()
        items = resp.json().get("items", [])
        for item in items:
            info = item.get("volumeInfo", {})
            desc = info.get("description", "")
            if desc:
                try:
                    if detect(desc) == "en":
                        return {
                            "description": desc,
                            "cover_url":   info.get("imageLinks", {}).get("thumbnail", ""),
                        }
                except:
                    continue
        return None
    except:
        return None

# ── Load checkpoint if exists ──────────────────────────────────────────────
if GB_CKPT_PATH.exists():
    with open(GB_CKPT_PATH) as f:
        gb_results = json.load(f)
    print(f"Resuming — {len(gb_results):,} already done")
else:
    gb_results = {}
    print("Starting fresh")

# ── Only process books missing descriptions ────────────────────────────────
needs_desc = merged[merged["description"].apply(
    lambda x: not bool(str(x).strip()) or str(x).strip() == "nan"
)].copy()

remaining = needs_desc[~needs_desc["source_url"].isin(gb_results.keys())]
print(f"Total needing descriptions: {len(needs_desc):,}")
print(f"Already fetched:            {len(needs_desc) - len(remaining):,}")
print(f"Remaining to fetch:         {len(remaining):,}")

stats = {"filled": 0, "not_found": 0, "error": 0}

for i, (idx, row) in enumerate(tqdm(
    remaining.iterrows(),
    total=len(remaining),
    desc="Google Books",
    unit="book"
), 1):
    key    = row["source_url"]
    title  = row["title"]
    author = row["author"]

    try:
        result = fetch_google_books_description(title, author)
        gb_results[key] = result if result else {}
        if result:
            stats["filled"] += 1
        else:
            stats["not_found"] += 1
    except Exception as e:
        gb_results[key] = {}
        stats["error"] += 1

    if i % 100 == 0:
        with open(GB_CKPT_PATH, "w") as f:
            json.dump(gb_results, f)

    time.sleep(0.5)

# Final checkpoint save
with open(GB_CKPT_PATH, "w") as f:
    json.dump(gb_results, f)

print(f"\n✅ Google Books enrichment complete")
print(f"   Filled:    {stats['filled']:,}")
print(f"   Not found: {stats['not_found']:,}")
print(f"   Errors:    {stats['error']:,}")

API key loaded: ✅
Starting fresh
Total needing descriptions: 2,174
Already fetched:            0
Remaining to fetch:         2,174


Google Books:   0%|          | 0/2174 [00:00<?, ?book/s]